In [3]:
!pip install librosa
!pip install panns-inference

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [panns-inference]


In [4]:
!pip install -qU pinecone-client==3.1.0 panns-inference datasets librosa

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


In [5]:
!pip install remotezip

In [6]:
import os, glob, zipfile, tarfile, time, json
import numpy as np, librosa, requests
from tqdm.auto import tqdm
from dotenv import load_dotenv, find_dotenv
from pinecone import Pinecone, ServerlessSpec
from panns_inference import AudioTagging, labels
from remotezip import RemoteZip

In [7]:
load_dotenv(find_dotenv(), override=True)
PANNS_DIR = os.path.expanduser("~/panns_data")
DATA_DIR = "data/music"
INDEX_NAME = "audio-search-demo"
NAMESPACE = "music"          # separate from the default namespace
MAX_TRACKS = 100             # CPU embedding is slow; raise later
CLIP_SECONDS = 30
SR = 32000                   # PANNs sample rate
ZENODO_RECORD = "5794629"    # Artificial multitrack audio data
AUDIO_EXT = (".wav", ".mp3", ".flac", ".ogg")
os.makedirs(DATA_DIR, exist_ok=True)

In [8]:
# 2. Build the sound database: download from Zenodo
def download(url, dest):
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True, desc=os.path.basename(dest)) as bar:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk); bar.update(len(chunk))

def extract(path):
    if path.endswith(".zip"):
        with zipfile.ZipFile(path) as z: z.extractall(DATA_DIR)
    elif path.endswith((".tar", ".tar.gz", ".tgz")):
        with tarfile.open(path) as t: t.extractall(DATA_DIR)

def find_audio():
    out = []
    for e in AUDIO_EXT:
        out += glob.glob(f"{DATA_DIR}/**/*{e}", recursive=True)
    return sorted(out)

if not find_audio():
    rec = requests.get(f"https://zenodo.org/api/records/{ZENODO_RECORD}", timeout=60).json()
    print("Title:", rec["metadata"]["title"])
    files = rec["files"]
    for f in files:
        print(f'  {f["key"]}  {f["size"]} bytes')
    # smallest file first (archive or audio); change the choice here if needed
    f = next(x for x in files if x["key"] == "0001-1000-audio-mixes.zip")
    with RemoteZip(f["links"]["self"]) as z:
        names = [n for n in z.namelist() if n.lower().endswith(AUDIO_EXT)]
        print("Audio files inside the zip:", len(names))
        step = max(1, len(names) // MAX_TRACKS)
        for n in tqdm(names[::step][:MAX_TRACKS], desc="Fetching tracks"):
            z.extract(n, DATA_DIR)

tracks = find_audio()
print("Audio files found:", len(tracks))
assert tracks, "No audio found: put mp3/wav files in data/music/ and rerun."
# spread the sample across the whole list instead of taking only the first files
step = max(1, len(tracks) // MAX_TRACKS)
tracks = tracks[::step][:MAX_TRACKS]
print("Tracks used:", len(tracks))

Title: AAM: Artificial Audio Multitracks Dataset
  1001-2000-midis.zip  3544696 bytes
  2001-3000-audio-multitracks.zip  56081467648 bytes
  1001-2000-audio-multitracks.zip  55315874129 bytes
  2001-3000-midis.zip  3835784 bytes
  2001-3000-annotations-v1.1.0.zip  8745087 bytes
  1001-2000-audio-mixes.zip  14730539954 bytes
  1001-2000-annotations-v1.1.0.zip  8601012 bytes
  0001-1000-midis.zip  2108953 bytes
  info.zip  11464390 bytes
  2001-3000-audio-mixes.zip  14778424664 bytes
  0001-1000-audio-multitracks.zip  53893757263 bytes
  0001-1000-audio-mixes.zip  14710551938 bytes
  0001-1000-annotations-v1.1.0.zip  8451769 bytes
Audio files inside the zip: 1000


Fetching tracks:   0%|          | 0/100 [00:00<?, ?it/s]

Audio files found: 100
Tracks used: 100


In [9]:
# 3. Embedding model (PANNs Cnn14, 2048-dim)
model = AudioTagging(checkpoint_path=os.path.join(PANNS_DIR, "Cnn14_mAP=0.431.pth"), device="cpu")

def embed(path):
    y, _ = librosa.load(path, sr=SR, mono=True, duration=CLIP_SECONDS)
    clip, emb = model.inference(y[None, :])
    top = np.argsort(clip[0])[::-1][:3]
    tags = [labels[i] for i in top]
    return emb[0], tags

Checkpoint path: /root/panns_data/Cnn14_mAP=0.431.pth
Using CPU.


In [10]:
# 4. Pinecone index
from getpass import getpass
os.environ["PINECONE_API_KEY"] = getpass("Pinecone API key: ")
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"].strip())
if INDEX_NAME not in pc.list_indexes().names():
    pc.create_index(INDEX_NAME, dimension=2048, metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))
    while not pc.describe_index(INDEX_NAME).status["ready"]:
        time.sleep(1)
index = pc.Index(INDEX_NAME)

Pinecone API key: ··········


In [11]:
# 5. Embed every track and upsert with metadata
vectors, cache = [], {}
for i, path in enumerate(tqdm(tracks, desc="Embedding")):
    try:
        v, tags = embed(path)
    except Exception as e:
        print("skip", path, e); continue
    cache[f"track-{i}"] = v
    vectors.append({"id": f"track-{i}", "values": v.tolist(),
                    "metadata": {"name": os.path.basename(path), "path": path, "tags": ", ".join(tags)}})
for i in range(0, len(vectors), 50):
    index.upsert(vectors=vectors[i:i + 50], namespace=NAMESPACE)

time.sleep(10)   # let Pinecone make the vectors queryable
print(index.describe_index_stats())


Embedding:   0%|          | 0/100 [00:00<?, ?it/s]

{'dimension': 2048,
 'index_fullness': 0.0,
 'namespaces': {'music': {'vector_count': 100}},
 'total_vector_count': 100}


In [12]:
# 6. Recommend from the user's preferences (liked tracks)
def recommend(liked_ids, k=5):
    q = np.mean([cache[i] for i in liked_ids], axis=0).tolist()
    res = index.query(vector=q, top_k=k + len(liked_ids), namespace=NAMESPACE, include_metadata=True)
    out = [m for m in res.matches if m.id not in liked_ids][:k]
    for m in out:
        print(f"{m.id:>10}  score={m.score:.4f}  {m.metadata['name']}  [{m.metadata['tags']}]")
    return out

liked = ["track-0", "track-1"]     # <- the user's preferences
print("Liked:")
for i in liked:
    meta = [v for v in vectors if v["id"] == i][0]["metadata"]
    print(f"  {i}  {meta['name']}  [{meta['tags']}]")
print("\nRecommendations:")
recs = recommend(liked, k=5)

Liked:
  track-0  0001_mix.flac  [Music, Musical instrument, Guitar]
  track-1  0011_mix.flac  [Music, Musical instrument, Piano]

Recommendations:
  track-32  score=0.9284  0321_mix.flac  [Music, Musical instrument, Keyboard (musical)]
   track-4  score=0.9263  0041_mix.flac  [Music, Musical instrument, Piano]
  track-60  score=0.9219  0601_mix.flac  [Music, Musical instrument, Piano]
  track-26  score=0.9213  0261_mix.flac  [Music, Musical instrument, Keyboard (musical)]
  track-88  score=0.9194  0881_mix.flac  [Music, Musical instrument, Guitar]


In [13]:
from google.colab import files
files.upload()   # a file picker opens, choose miaow_16k.wav
os.makedirs("data", exist_ok=True)
os.replace("miaow_16k.wav", "data/miaow_16k.wav")
print(os.path.exists("data/miaow_16k.wav"))   # should print True

Saving miaow_16k.wav to miaow_16k (1).wav
True


In [17]:
import glob
print(glob.glob("**/miaow*", recursive=True))

v, tags = embed("miaow_16k (1).wav")   # replace with the path printed above
print("Miaow tags:", tags)
res = index.query(vector=v.tolist(), top_k=3, namespace=NAMESPACE, include_metadata=True)
for m in res.matches:
    print(m.id, round(m.score, 4), m.metadata["name"], m.metadata["tags"])

['miaow_16k (1).wav', 'data/miaow_16k.wav']
Miaow tags: ['Animal', 'Cat', 'Domestic animals, pets']
track-15 0.5831 0151_mix.flac Music, Musical instrument, New-age music
track-72 0.55 0721_mix.flac Music, Musical instrument, Guitar
track-3 0.5277 0031_mix.flac Music, Musical instrument, Drum
